# Step 3: Model Building & Training
## Attention-CNN-LSTM for Intrusion Detection

This notebook implements the exact architecture from the paper:
- **CNN layers** → spatial feature extraction
- **Batch Normalization** → stable training
- **LSTM layer** → temporal sequence modeling
- **Attention mechanism** → focus on important features
- **Dense + Softmax** → final classification

> **Make sure you completed Notebook 2 before running this.**

---
## Cell 1: Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import json
import os
import time
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

torch.manual_seed(42)
np.random.seed(42)
print('imports successful')

---
## Cell 2: Load Preprocessed Data

In [ ]:
X_train = np.load('data/processed/X_train.npy')
X_test  = np.load('data/processed/X_test.npy')
y_train = np.load('data/processed/y_train.npy')
y_test  = np.load('data/processed/y_test.npy')

with open('data/processed/metadata.json', 'r') as f:
    metadata = json.load(f)

NUM_FEATURES = metadata['num_features']
NUM_CLASSES  = metadata['num_classes']
CLASS_NAMES  = metadata['class_names']

print(f'X_train shape : {X_train.shape}')
print(f'X_test shape  : {X_test.shape}')
print(f'Num features  : {NUM_FEATURES}')
print(f'Num classes   : {NUM_CLASSES}')
print(f'Class names   : {CLASS_NAMES}')
print('Data loaded successfully')

---
## Cell 3: PyTorch Dataset and DataLoader

In [ ]:
class IntrusionDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# Hyperparameters from Table 2 of the paper
BATCH_SIZE = 128
EPOCHS     = 10
LR         = 0.001
DROPOUT    = 0.3

train_dataset = IntrusionDataset(X_train, y_train)
test_dataset  = IntrusionDataset(X_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f'Batch size      : {BATCH_SIZE}')
print(f'Training batches: {len(train_loader)}')
print(f'Test batches    : {len(test_loader)}')
print('DataLoaders created')

---
## Cell 4: Attention Mechanism

Implements Equation 10 from the paper:

**alpha_i = exp(e_i) / sum(exp(e_j))**

In [ ]:
class SelfAttention(nn.Module):
    """
    Self-Attention layer (Equation 10 from the paper).
    Input : (batch, seq_len, hidden_size)  from LSTM
    Output: (batch, hidden_size)           weighted context vector
    """
    def __init__(self, hidden_size):
        super(SelfAttention, self).__init__()
        self.attention_weights = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, lstm_output):
        # Compute raw attention scores
        e = self.attention_weights(lstm_output).squeeze(-1)   # (batch, seq_len)
        # Normalize with softmax
        alpha = torch.softmax(e, dim=1)                       # (batch, seq_len)
        # Weighted sum of LSTM outputs
        context = torch.sum(alpha.unsqueeze(-1) * lstm_output, dim=1)  # (batch, hidden)
        return context, alpha


print('SelfAttention module defined')

---
## Cell 5: Full Attention-CNN-LSTM Model

Architecture (Figure 4 from the paper):
```
Input -> CNN -> BatchNorm -> MaxPool
      -> CNN -> BatchNorm -> MaxPool
      -> LSTM -> Dropout
      -> Attention
      -> Dense -> Softmax
```

In [ ]:
class AttentionCNNLSTM(nn.Module):
    """
    Hybrid Attention-CNN-LSTM model.
    Hyperparameters match Table 2 of the paper.
    """
    def __init__(self, num_features, num_classes,
                 cnn_filters_1=64, cnn_filters_2=128,
                 lstm_units=64, dropout=0.3):
        super(AttentionCNNLSTM, self).__init__()

        # CNN Block 1: 64 filters, kernel 3
        self.conv1  = nn.Conv1d(1, cnn_filters_1, kernel_size=3, padding=1)
        self.bn1    = nn.BatchNorm1d(cnn_filters_1)
        self.relu1  = nn.ReLU()
        self.pool1  = nn.MaxPool1d(kernel_size=2, stride=2)

        # CNN Block 2: 128 filters, kernel 3
        self.conv2  = nn.Conv1d(cnn_filters_1, cnn_filters_2, kernel_size=3, padding=1)
        self.bn2    = nn.BatchNorm1d(cnn_filters_2)
        self.relu2  = nn.ReLU()
        self.pool2  = nn.MaxPool1d(kernel_size=2, stride=2)

        # LSTM: 64 units
        self.lstm    = nn.LSTM(cnn_filters_2, lstm_units, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(dropout)

        # Attention layer
        self.attention = SelfAttention(hidden_size=lstm_units)

        # Dense output layers
        self.fc      = nn.Linear(lstm_units, 64)
        self.relu_fc = nn.ReLU()
        self.out     = nn.Linear(64, num_classes)

    def forward(self, x):
        # x: (batch, 1, num_features)
        out = self.pool1(self.relu1(self.bn1(self.conv1(x))))   # (batch, 64,  F//2)
        out = self.pool2(self.relu2(self.bn2(self.conv2(out)))) # (batch, 128, F//4)
        out = out.permute(0, 2, 1)                              # (batch, seq, 128)
        out, _ = self.lstm(out)                                 # (batch, seq, 64)
        out = self.dropout(out)
        context, alpha = self.attention(out)                    # (batch, 64)
        out = self.relu_fc(self.fc(context))                    # (batch, 64)
        return self.out(out)                                    # (batch, num_classes)


model = AttentionCNNLSTM(
    num_features=NUM_FEATURES,
    num_classes=NUM_CLASSES,
    cnn_filters_1=64,
    cnn_filters_2=128,
    lstm_units=64,
    dropout=DROPOUT
).to(device)

print('Model created!')
print(model)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

# Quick forward pass test
dummy = torch.randn(4, 1, NUM_FEATURES).to(device)
with torch.no_grad():
    test_out = model(dummy)
print(f'Input shape : {dummy.shape}')
print(f'Output shape: {test_out.shape}  <- should be (4, {NUM_CLASSES})')
print('Forward pass works!')

---
## Cell 6: Loss Function and Optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=2, factor=0.5, verbose=True
)

print(f'Loss      : CrossEntropyLoss')
print(f'Optimizer : Adam (lr={LR})')
print(f'Scheduler : ReduceLROnPlateau')

---
## Cell 7: Training and Evaluation Functions

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss    = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
        correct    += (outputs.argmax(dim=1) == y_batch).sum().item()
        total      += X_batch.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs     = model(X_batch)
            loss        = criterion(outputs, y_batch)
            total_loss += loss.item() * X_batch.size(0)
            correct    += (outputs.argmax(dim=1) == y_batch).sum().item()
            total      += X_batch.size(0)
    return total_loss / total, correct / total


print('Training functions defined')

---
## Cell 8: Training Loop

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}
best_test_acc = 0.0
best_epoch    = 0

print(f'Starting training for {EPOCHS} epochs...')
print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Train Acc":>9} | {"Test Loss":>9} | {"Test Acc":>8} | {"Time":>6}')
print('-' * 65)

for epoch in range(1, EPOCHS + 1):
    start = time.time()

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    test_loss,  test_acc  = evaluate(model, test_loader, criterion, device)
    scheduler.step(test_loss)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)

    if test_acc > best_test_acc:
        best_test_acc = test_acc
        best_epoch    = epoch
        torch.save(model.state_dict(), 'models/best_model.pth')
        saved = ' <- Best saved'
    else:
        saved = ''

    elapsed = time.time() - start
    print(f'{epoch:>6} | {train_loss:>10.4f} | {train_acc*100:>8.2f}% | {test_loss:>9.4f} | {test_acc*100:>7.2f}%{saved} | {elapsed:>5.1f}s')

print('-' * 65)
print(f'Training complete!')
print(f'Best test accuracy: {best_test_acc*100:.2f}% at epoch {best_epoch}')
print(f'Model saved to: models/best_model.pth')

---
## Cell 9: Plot Training Curves

In [ ]:
epochs_range = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, [a*100 for a in history['train_acc']], 'b-o', label='Train Accuracy')
axes[0].plot(epochs_range, [a*100 for a in history['test_acc']],  'r-o', label='Test Accuracy')
axes[0].set_title('Model Accuracy Over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy (%)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss')
axes[1].plot(epochs_range, history['test_loss'],  'r-o', label='Test Loss')
axes[1].set_title('Model Loss Over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/training_curves.png', dpi=150)
plt.show()
print('Training curves saved to results/training_curves.png')
print('Next step: Run notebook -> 04_evaluation.ipynb')